# Loss-Term Scale / Weight Calculation

Computes `weight_pixel`, `weight_patch`, `weight_spectral` for `combined_loss_fn`
directly from data (population variance of the true `v` field), replacing the
old `load_solo_scale` approach (which depended on stale/possibly-under-converged
solo training runs).

**Method:** for each loss term, the scale is the population variance of the
quantity that term measures deviations of, evaluated on the true (`v`) patches
alone:

- `scale_pixel`    = population variance of raw pixel values
- `scale_patch`    = population variance of per-patch mean values
- `scale_spectral` = population variance of log(radial power spectrum + eps)

Then `weight_x = 1 / scale_x`. All three are computed from the **same** shared
sample of patches, drawn from **v only** (tau is never compared against in any
loss term, so it isn't needed here).

**Data used:** 3 randomly chosen realizations from each of the three shell
maps `v0`, `v1`, `v2` (9 realizations total, all patches from each).

This notebook is standalone — the small set of pipeline functions it needs
(patch extraction, radial power spectrum binning) are reproduced here rather
than imported, so it can run independently of the training script.

## Imports & configuration

In [10]:
import os
import numpy as np
import healpy as hp

RUN_SEED = 0
rng = np.random.default_rng(RUN_SEED)

DATA_DIR = "/mnt/beegfs/scoulombe"

NSIDE_MAP   = 2048
NSIDE_PATCH = 256
LMAX        = 3 * NSIDE_MAP - 1

N_SPEC_BINS   = 22
SPEC_LOSS_EPS = 1e-10

N_REALIZATIONS_PER_SHELL = 32      # total realizations available per shell map
N_SAMPLE_REALIZATIONS    = 3       # how many to randomly draw from EACH shell

V_FIELD_NAMES = ["v0", "v1", "v2"]

SMOOTH_WINDOW = 5  # must match generate_realizations.py naming

V_FILES = {
    name: os.path.join(
        DATA_DIR,
        f"realizations_{name}_nside{NSIDE_MAP}_n{N_REALIZATIONS_PER_SHELL}_norm_smooth{SMOOTH_WINDOW}.npy",
    )
    for name in V_FIELD_NAMES
}

for name, path in V_FILES.items():
    print(name, "->", path, "exists:", os.path.exists(path))

OUT_FILE = os.path.join(DATA_DIR, "binned_loss_scale.npz")
print("Will save weights/scales to:", OUT_FILE)

v0 -> /mnt/beegfs/scoulombe/realizations_v0_nside2048_n32_norm_smooth5.npy exists: True
v1 -> /mnt/beegfs/scoulombe/realizations_v1_nside2048_n32_norm_smooth5.npy exists: True
v2 -> /mnt/beegfs/scoulombe/realizations_v2_nside2048_n32_norm_smooth5.npy exists: True
Will save weights/scales to: /mnt/beegfs/scoulombe/binned_loss_scale.npz


## Patch extraction (reproduced from the training pipeline)

Same NEST-ordering face-splitting logic as `precompute_face_indices` /
`get_face_2d` / `extract_all_patches_from_map` in the main training script,
so the patches here are extracted identically to how the model sees them
during training.

In [11]:
def precompute_face_indices(nside):
    npix = hp.nside2npix(nside)
    ipix = np.arange(npix, dtype=np.int32)
    x, y, f = hp.pix2xyf(nside, ipix, nest=True)
    face_idx = {}
    for face_id in range(12):
        mask = f == face_id
        face_idx[face_id] = (ipix[mask].astype(np.int32), x[mask].astype(np.int32), y[mask].astype(np.int32))
    return face_idx

FACE_IDX = precompute_face_indices(NSIDE_MAP)

def get_face_2d(map_nest, nside, face_id, face_idx):
    ipix_f, x_f, y_f = face_idx[face_id]
    face_img = np.full((nside, nside), np.nan, dtype=map_nest.dtype)
    face_img[y_f, x_f] = map_nest[ipix_f]
    assert not np.isnan(face_img).any(), f"Face {face_id} has unfilled pixels — reshape failed"
    return face_img

def extract_all_patches_from_map(map_nest, nside_map, nside_patch, face_idx=FACE_IDX):
    n_per_side = nside_map // nside_patch
    n_patches = 12 * n_per_side * n_per_side
    out = np.empty((n_patches, nside_patch, nside_patch), dtype=map_nest.dtype)
    idx = 0
    for face_id in range(12):
        face_img = get_face_2d(map_nest, nside_map, face_id, face_idx)
        for pr in range(n_per_side):
            for pc in range(n_per_side):
                out[idx] = face_img[pr*nside_patch:(pr+1)*nside_patch,
                                     pc*nside_patch:(pc+1)*nside_patch]
                idx += 1
    return out

print("Patch extraction functions ready")

Patch extraction functions ready


## Radial power spectrum (reproduced from the training pipeline)

Identical to `get_radial_bins` / `radial_power_spectrum` in the training
script — same log-spaced radial binning, same DC-into-bin-0 behavior.

In [12]:
_RADIAL_BIN_CACHE = {}

def get_radial_bins(patch_size, n_bins):
    key = (patch_size, n_bins)
    if key in _RADIAL_BIN_CACHE:
        return _RADIAL_BIN_CACHE[key]

    freqs_y = np.fft.fftfreq(patch_size)
    freqs_x = np.fft.rfftfreq(patch_size)
    ky, kx = np.meshgrid(freqs_y, freqs_x, indexing='ij')
    k = np.sqrt(kx**2 + ky**2)
    k_flat = k.flatten()

    nonzero = k_flat[k_flat > 0]
    kmin = nonzero.min()
    kmax = k_flat.max()
    bin_edges = np.logspace(np.log10(kmin), np.log10(kmax), n_bins + 1)

    bin_idx = np.digitize(k_flat, bin_edges) - 1
    bin_idx = np.clip(bin_idx, 0, n_bins - 1).astype(np.int64)

    _RADIAL_BIN_CACHE[key] = bin_idx
    return bin_idx

def radial_power_spectrum(patch_batch, n_bins=N_SPEC_BINS):
    """patch_batch: (B, H, W) numpy array. Returns (B, n_bins)."""
    B, H, W = patch_batch.shape
    fft = np.fft.rfft2(patch_batch)
    power = fft.real**2 + fft.imag**2

    bin_idx = get_radial_bins(H, n_bins)
    power_flat = power.reshape(B, -1)

    binned = np.zeros((B, n_bins), dtype=power.dtype)
    counts = np.zeros(n_bins, dtype=power.dtype)
    np.add.at(counts, bin_idx, 1.0)
    counts = np.clip(counts, 1.0, None)

    for b in range(n_bins):
        mask = bin_idx == b
        if mask.any():
            binned[:, b] = power_flat[:, mask].mean(axis=1)

    return binned

print("Spectral functions ready")

Spectral functions ready


## Draw the shared random sample: 3 realizations from each of v0, v1, v2

All downstream scale calculations use patches from this exact set of 9
realizations — the "same sample of patches" requirement.

In [13]:
chosen_realizations = {}
for name in V_FIELD_NAMES:
    chosen = rng.choice(N_REALIZATIONS_PER_SHELL, size=N_SAMPLE_REALIZATIONS, replace=False)
    chosen_realizations[name] = sorted(chosen.tolist())
    print(f"{name}: realizations {chosen_realizations[name]}")

print(f"\nTotal realizations sampled: {sum(len(v) for v in chosen_realizations.values())}")

v0: realizations [16, 19, 25]
v1: realizations [0, 1, 2]
v2: realizations [16, 19, 28]

Total realizations sampled: 9


## Extract patches from the sampled realizations

Maps are stored RING-ordered (`hp.synfast` default), so each is reordered to
NEST before patch extraction, matching the training pipeline's
`hp.reorder(v_ring, r2n=True)` step.

In [14]:
all_patch_batches = []

for name in V_FIELD_NAMES:
    mmap = np.load(V_FILES[name], mmap_mode="r")
    for real_idx in chosen_realizations[name]:
        v_ring = np.array(mmap[real_idx], dtype=np.float32)
        v_nest = hp.reorder(v_ring, r2n=True)
        patches = extract_all_patches_from_map(v_nest, NSIDE_MAP, NSIDE_PATCH)
        all_patch_batches.append(patches)
        print(f"  {name} realization {real_idx}: extracted {patches.shape[0]} patches")
        del v_ring, v_nest, patches
    del mmap

v_patches = np.concatenate(all_patch_batches, axis=0)
del all_patch_batches
print(f"\nTotal patches sampled: {v_patches.shape[0]}  (shape: {v_patches.shape})")

  v0 realization 16: extracted 768 patches
  v0 realization 19: extracted 768 patches
  v0 realization 25: extracted 768 patches
  v1 realization 0: extracted 768 patches
  v1 realization 1: extracted 768 patches
  v1 realization 2: extracted 768 patches
  v2 realization 16: extracted 768 patches
  v2 realization 19: extracted 768 patches
  v2 realization 28: extracted 768 patches

Total patches sampled: 6912  (shape: (6912, 256, 256))


## Compute the three scales (population variance, `ddof=0`)

All three are computed from `v_patches` only — the same shared sample.

In [15]:
# --- pixel scale: population variance of raw pixel values ---
scale_pixel = np.var(v_patches, ddof=0)

# --- patch scale: population variance of per-patch mean values ---
patch_means = v_patches.mean(axis=(1, 2))
scale_patch = np.var(patch_means, ddof=0)

# --- spectral scale: population variance of log(power + eps) ---
P_true = radial_power_spectrum(v_patches, n_bins=N_SPEC_BINS)
log_true = np.log(P_true + SPEC_LOSS_EPS)
scale_spectral = np.var(log_true, ddof=0)

print(f"scale_pixel    = {scale_pixel:.6e}")
print(f"scale_patch    = {scale_patch:.6e}")
print(f"scale_spectral = {scale_spectral:.6e}")

scale_pixel    = 1.217137e+00
scale_patch    = 4.168061e-03
scale_spectral = 3.150008e+00


In [16]:
weight_pixel    = 1.0 / scale_pixel
weight_patch    = 1.0 / scale_patch
weight_spectral = 1.0 / scale_spectral

print(f"weight_pixel    = {weight_pixel:.6e}")
print(f"weight_patch    = {weight_patch:.6e}")
print(f"weight_spectral = {weight_spectral:.6e}")

weight_pixel    = 8.216000e-01
weight_patch    = 2.399197e+02
weight_spectral = 3.174596e-01


## Self-consistency check

By construction, `weight_x * scale_x` must equal `1.0` for each term — this
just verifies there's no bug in the pairing between a weight and its scale
(e.g. transposed variables, wrong array reused). It does **not** check
equal contribution against real model predictions; it's a sanity check on
this notebook's own arithmetic.

In [17]:
CHECK_TOL = 1e-6

checks = {
    "pixel":    (weight_pixel,    scale_pixel),
    "patch":    (weight_patch,    scale_patch),
    "spectral": (weight_spectral, scale_spectral),
}

print("Self-consistency check: weight_x * scale_x should equal 1.0\n")
all_ok = True
for label, (w, s) in checks.items():
    product = w * s
    ok = abs(product - 1.0) < CHECK_TOL
    all_ok &= ok
    status = "OK" if ok else "FAILED"
    print(f"  {label:9s}: weight * scale = {product:.8f}   [{status}]")

if all_ok:
    print("\nAll three terms pass self-consistency: each weighted term contributes")
    print("~1.0 on the true data used to derive its own scale (by construction).")
else:
    raise AssertionError("Self-consistency check failed — a weight/scale pair does not match. "
                          "Check for a bug before saving/using these weights.")

Self-consistency check: weight_x * scale_x should equal 1.0

  pixel    : weight * scale = 1.00000000   [OK]
  patch    : weight * scale = 1.00000000   [OK]
  spectral : weight * scale = 1.00000000   [OK]

All three terms pass self-consistency: each weighted term contributes
~1.0 on the true data used to derive its own scale (by construction).


## Save weights and scales

In [18]:
np.savez(
    OUT_FILE,
    weight_pixel=weight_pixel,
    weight_patch=weight_patch,
    weight_spectral=weight_spectral,
    scale_pixel=scale_pixel,
    scale_patch=scale_patch,
    scale_spectral=scale_spectral,
    n_spec_bins=N_SPEC_BINS,
    spec_loss_eps=SPEC_LOSS_EPS,
    nside_map=NSIDE_MAP,
    nside_patch=NSIDE_PATCH,
    n_realizations_per_shell_sampled=N_SAMPLE_REALIZATIONS,
    v_field_names=np.array(V_FIELD_NAMES),
    chosen_realizations_v0=np.array(chosen_realizations["v0"]),
    chosen_realizations_v1=np.array(chosen_realizations["v1"]),
    chosen_realizations_v2=np.array(chosen_realizations["v2"]),
    run_seed=RUN_SEED,
    n_patches_sampled=v_patches.shape[0],
)

print(f"Saved to {OUT_FILE}")
print("\nTo use in the training script, replace the load_solo_scale(...) calls with:\n")
print('    _scales = np.load(os.path.join(DATA_DIR, "binned_loss_scale.npz"))')
print('    weight_pixel    = float(_scales["weight_pixel"])')
print('    weight_patch    = float(_scales["weight_patch"])')
print('    weight_spectral = float(_scales["weight_spectral"])')

Saved to /mnt/beegfs/scoulombe/binned_loss_scale.npz

To use in the training script, replace the load_solo_scale(...) calls with:

    _scales = np.load(os.path.join(DATA_DIR, "binned_loss_scale.npz"))
    weight_pixel    = float(_scales["weight_pixel"])
    weight_patch    = float(_scales["weight_patch"])
    weight_spectral = float(_scales["weight_spectral"])
